# Confluence Data Extraction

This notebook provides a structured approach to extract data from Confluence using the REST API. We'll use Python libraries to fetch, process, and analyze data from your Confluence instance.

## Install and Import Required Libraries
First, let's install the necessary packages and import them for our work.

In [4]:
# Install required packages (uncomment if needed)
# !pip install requests pandas python-dotenv beautifulsoup4 html2text

In [5]:
# Import required libraries
import requests
import pandas as pd
import json
import os
from dotenv import load_dotenv
import base64
from requests.auth import HTTPBasicAuth
import time
from datetime import datetime
import html2text
import re

## Configure Confluence API Access

Set up the connection details for your Confluence instance. For security, we'll use environment variables or a .env file.

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Set up Confluence API connection details
# You can either set these as environment variables or replace with your actual values
CONFLUENCE_URL = os.getenv('CONFLUENCE_URL', 'CONFLUENCE_URL')
CONFLUENCE_USERNAME = os.getenv('CONFLUENCE_USERNAME', 'CONFLUENCE_USERNAME')
CONFLUENCE_API_TOKEN = os.getenv('CONFLUENCE_API_TOKEN', CONFLUENCE_API_TOKEN')

# Check if credentials are properly loaded
if not all([CONFLUENCE_URL, CONFLUENCE_USERNAME, CONFLUENCE_API_TOKEN]):
    print("⚠️ Warning: Confluence credentials not fully configured.")
    print("Please set the CONFLUENCE_URL, CONFLUENCE_USERNAME, and CONFLUENCE_API_TOKEN environment variables.")
else:
    print("✅ Confluence credentials loaded successfully.")
    print(f"Configured to connect to: {CONFLUENCE_URL}")

✅ Confluence credentials loaded successfully.
Configured to connect to: https://ohmgym.atlassian.net


## Define Helper Functions

Let's create some helper functions to interact with the Confluence API.

In [ ]:
def create_confluence_session():
    """Create and return a session object for Confluence API calls with proper authentication"""
    session = requests.Session()
    session.auth = HTTPBasicAuth(CONFLUENCE_USERNAME, CONFLUENCE_API_TOKEN)
    session.headers.update({
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    })
    return session

def get_space_info(session, space_key):
    """Get information about a specific space"""
    url = f"{CONFLUENCE_URL}/wiki/rest/api/space/{space_key}"
    response = session.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching space info: {response.status_code}")
        print(response.text)
        return None

def get_all_spaces(session):
    """Get information about all spaces the user has access to"""
    spaces = []
    start = 0
    limit = 100
    while True:
        url = f"{CONFLUENCE_URL}/wiki/rest/api/space?start={start}&limit={limit}"
        response = session.get(url)
        if response.status_code != 200:
            print(f"Error fetching spaces: {response.status_code}")
            print(response.text)
            break
            
        data = response.json()
        results = data.get('results', [])
        if not results:
            break
            
        spaces.extend(results)
        if len(results) < limit:
            break
            
        start += limit
        time.sleep(0.5) 
    
    return spaces

def get_pages_in_space(session, space_key, expand=None):
    """Get all pages in a specific space"""
    pages = []
    start = 0
    limit = 50
    expand_param = f"&expand={expand}" if expand else ""
    
    while True:
        url = f"{CONFLUENCE_URL}/wiki/rest/api/space/{space_key}/content/page?start={start}&limit={limit}{expand_param}"
        response = session.get(url)
        if response.status_code != 200:
            print(f"Error fetching pages: {response.status_code}")
            print(response.text)
            break
            
        data = response.json()
        results = data.get('results', [])
        if not results:
            break
            
        pages.extend(results)
        if len(results) < limit:
            break
            
        start += limit
        time.sleep(0.5)  # Be nice to the API
    
    return pages

def get_page_content(session, page_id, expand='body.storage'):
    """Get detailed content for a specific page"""
    url = f"{CONFLUENCE_URL}/wiki/rest/api/content/{page_id}?expand={expand}"
    response = session.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching page content: {response.status_code}")
        print(response.text)
        return None

def extract_text_from_html(html_content):
    """Convert HTML content to plain text"""
    h = html2text.HTML2Text()
    h.ignore_links = False
    h.ignore_images = False
    h.ignore_tables = False
    return h.handle(html_content)

## Connect to Confluence API

Let's establish a connection to your Confluence instance.

In [8]:
# Create a session for Confluence API
try:
    session = create_confluence_session()
    print("Session created successfully.")
except Exception as e:
    print(f"Error creating session: {e}")

Session created successfully.


## List Available Spaces

Let's retrieve a list of available spaces in your Confluence instance.

In [9]:
# Get all available spaces
spaces = get_all_spaces(session)

# Create a DataFrame for easy viewing
if spaces:
    spaces_df = pd.DataFrame([{
        'key': space['key'],
        'name': space['name'],
        'type': space['type'],
        'description': space.get('description', {}).get('plain', {}).get('value', 'No description')
    } for space in spaces])
    
    display(spaces_df)
    print(f"Found {len(spaces)} spaces.")
else:
    print("No spaces found or error occurred.")

,key,name,type,description
0,~712020e4fab5d9f41d4c3c82332217b200b0a4,Chris Weinreich,personal,No description
1,Founders,Founders,global,No description
2,KB,Knowledge base,global,No description
3,~7120201a4dea3144c247179c61f82e27bbdb1f,ohm,personal,No description
4,O2,OhmGym 2.0,global,No description
5,SD,Software Development,global,No description


Found 6 spaces.


## Extract Pages from a Specific Space

Now, let's extract pages from a specific space. You'll need to provide the space key from the list above.

In [10]:
# Set the space key you want to work with
SPACE_KEY = "Founders"  # Replace with your actual space key

# Get all pages in the space
pages = get_pages_in_space(session, SPACE_KEY)

# Create a DataFrame for the pages
if pages:
    pages_df = pd.DataFrame([{
        'id': page['id'],
        'title': page['title'],
        'type': page['type'],
        'status': page.get('status', 'unknown'),
        'created': page.get('history', {}).get('createdDate', 'unknown'),
        'updated': page.get('history', {}).get('lastUpdated', {}).get('when', 'unknown'),
        'version': page.get('version', {}).get('number', 0),
    } for page in pages])
    
    # Convert datetime strings to datetime objects for sorting
    pages_df['created'] = pd.to_datetime(pages_df['created'], errors='ignore')
    pages_df['updated'] = pd.to_datetime(pages_df['updated'], errors='ignore')
    
    # Sort by most recently updated
    pages_df = pages_df.sort_values('updated', ascending=False)
    
    display(pages_df)
    print(f"Found {len(pages)} pages in space '{SPACE_KEY}'.")
else:
    print(f"No pages found in space '{SPACE_KEY}' or error occurred.")

/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1584582160.py:20: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  pages_df['created'] = pd.to_datetime(pages_df['created'], errors='ignore')
/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1584582160.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pages_df['created'] = pd.to_datetime(pages_df['created'], errors='ignore')
/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1584582160.py:21: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  pages_df['updated'] = pd.to_datetime(pages_df['updated'], errors='ignore')
/var/folders

,id,title,type,status,created,updated,version
0,4980952,Founders,page,current,unknown,unknown,0
1,5242881,Go To Market Sales Playbook,page,current,unknown,unknown,0
2,5308428,"Branding, Marketing, & Investors",page,current,unknown,unknown,0
3,5341200,Future Ideas of OhmGym,page,current,unknown,unknown,0
4,5341213,Angel Investor Pitch Notes,page,current,unknown,unknown,0
5,5472257,Business Plan,page,current,unknown,unknown,0
6,5832710,Competitors Systems and Services,page,current,unknown,unknown,0
7,7045121,Google Ads Metrics,page,current,unknown,unknown,0


Found 8 pages in space 'Founders'.


## Extract Content from Specific Pages

Now let's extract the content from specific pages. You can either extract from all pages or select specific ones based on criteria.

In [ ]:
# Option 1: Extract content from all pages (may take a while for large spaces)
def extract_all_page_content(pages, max_pages=None):
    """Extract content from multiple pages"""
    result = []
    
    # Limit the number of pages to extract if specified
    pages_to_extract = pages[:max_pages] if max_pages else pages
    
    total = len(pages_to_extract)
    print(f"Extracting content from {total} pages...")
    
    for i, page in enumerate(pages_to_extract, 1):
        page_id = page['id']
        print(f"Processing {i}/{total}: {page['title']} (ID: {page_id})")
        
        page_detail = get_page_content(session, page_id, expand='body.storage,version,ancestors,metadata')
        
        if page_detail and 'body' in page_detail and 'storage' in page_detail['body']:
            html_content = page_detail['body']['storage']['value']
            plain_text = extract_text_from_html(html_content)
            
            # Get ancestors (parent pages)
            ancestors = [a.get('title', 'Unknown') for a in page_detail.get('ancestors', [])]
            ancestor_path = ' > '.join(ancestors) if ancestors else 'Root'
            
            result.append({
                'id': page_id,
                'title': page['title'],
                'ancestors': ancestor_path,
                'url': f"{CONFLUENCE_URL}/wiki/spaces/{SPACE_KEY}/pages/{page_id}",
                'created_date': page_detail.get('history', {}).get('createdDate', 'unknown'),
                'last_updated': page_detail.get('version', {}).get('when', 'unknown'),
                'version': page_detail.get('version', {}).get('number', 0),
                'html_content': html_content,
                'plain_text': plain_text,
            })
            
            # Be nice to the API
            time.sleep(0.5)
        else:
            print(f"  ⚠️ Could not retrieve content for {page['title']}")
    
    return result

# # Option 2: Extract content from specific pages by ID
# def extract_page_by_id(page_id):
#     """Extract content from a single page by ID"""
#     page_detail = get_page_content(session, page_id, expand='body.storage,version,ancestors,metadata')
    
#     if page_detail and 'body' in page_detail and 'storage' in page_detail['body']:
#         html_content = page_detail['body']['storage']['value']
#         plain_text = extract_text_from_html(html_content)
        
#         # Get ancestors (parent pages)
#         ancestors = [a.get('title', 'Unknown') for a in page_detail.get('ancestors', [])]
#         ancestor_path = ' > '.join(ancestors) if ancestors else 'Root'
        
#         return {
#             'id': page_id,
#             'title': page_detail['title'],
#             'ancestors': ancestor_path,
#             'url': f"{CONFLUENCE_URL}/wiki/spaces/{page_detail.get('space', {}).get('key', 'unknown')}/pages/{page_id}",
#             'created_date': page_detail.get('history', {}).get('createdDate', 'unknown'),
#             'last_updated': page_detail.get('version', {}).get('when', 'unknown'),
#             'version': page_detail.get('version', {}).get('number', 0),
#             'html_content': html_content,
#             'plain_text': plain_text,
#         }
#     else:
#         print(f"  ⚠️ Could not retrieve content for page ID {page_id}")
#         return None

## Example: Extract and Analyze Content

Let's extract content from a few pages and perform some analysis.

In [12]:
# Extract content from the first 5 pages (adjust as needed)
if 'pages' in locals() and pages:
    page_contents = extract_all_page_content(pages, max_pages=5)
    
    if page_contents:
        # Convert to DataFrame for analysis
        content_df = pd.DataFrame(page_contents)
        
        # Display basic information about extracted pages
        print(f"Extracted content from {len(page_contents)} pages")
        display(content_df[['title', 'url', 'ancestors', 'last_updated', 'version']])
    else:
        print("No page contents were extracted.")
else:
    print("No pages available. Please run the previous cell to fetch pages first.")

Extracting content from 5 pages...
Processing 1/5: Founders (ID: 4980952)
Processing 2/5: Go To Market Sales Playbook (ID: 5242881)
Processing 2/5: Go To Market Sales Playbook (ID: 5242881)
Processing 3/5: Branding, Marketing, & Investors (ID: 5308428)
Processing 3/5: Branding, Marketing, & Investors (ID: 5308428)
Processing 4/5: Future Ideas of OhmGym (ID: 5341200)
Processing 4/5: Future Ideas of OhmGym (ID: 5341200)
Processing 5/5: Angel Investor Pitch Notes (ID: 5341213)
Processing 5/5: Angel Investor Pitch Notes (ID: 5341213)
Extracted content from 5 pages
Extracted content from 5 pages


,title,url,ancestors,last_updated,version
0,Founders,https://ohmgym.atlassian.net/wiki/spaces/Found...,Root,2024-01-09T23:07:33.715Z,1
1,Go To Market Sales Playbook,https://ohmgym.atlassian.net/wiki/spaces/Found...,Founders,2024-01-15T02:07:33.210Z,6
2,"Branding, Marketing, & Investors",https://ohmgym.atlassian.net/wiki/spaces/Found...,Founders,2024-01-15T18:45:03.308Z,3
3,Future Ideas of OhmGym,https://ohmgym.atlassian.net/wiki/spaces/Found...,Founders,2024-01-11T01:23:13.767Z,3
4,Angel Investor Pitch Notes,https://ohmgym.atlassian.net/wiki/spaces/Found...,Founders,2024-01-10T00:10:40.624Z,1


## Extract Page Metadata

Let's extract and analyze metadata about the pages in the space.

In [12]:
# Extract metadata from pages
def analyze_page_metadata(pages_df):
    """Analyze metadata from pages DataFrame"""
    if pages_df.empty:
        return "No pages data available for analysis."
    
    results = {
        'total_pages': len(pages_df),
        'page_versions': {
            'min': pages_df['version'].min(),
            'max': pages_df['version'].max(),
            'avg': pages_df['version'].mean()
        }
    }
    
    # Add date ranges if datetime columns are available
    if 'created' in pages_df.columns and pd.api.types.is_datetime64_any_dtype(pages_df['created']):
        results['date_range'] = {
            'earliest_created': pages_df['created'].min(),
            'latest_created': pages_df['created'].max(),
        }
    
    if 'updated' in pages_df.columns and pd.api.types.is_datetime64_any_dtype(pages_df['updated']):
        if 'date_range' not in results:
            results['date_range'] = {}
        results['date_range'].update({
            'earliest_updated': pages_df['updated'].min(),
            'latest_updated': pages_df['updated'].max(),
        })
        
        # Calculate activity by month
        if pd.api.types.is_datetime64_any_dtype(pages_df['updated']):
            pages_df['update_month'] = pages_df['updated'].dt.to_period('M')
            monthly_updates = pages_df.groupby('update_month').size()
            results['monthly_activity'] = monthly_updates.to_dict()
    
    return results

# If pages_df exists, analyze its metadata
if 'pages_df' in locals() and not pages_df.empty:
    metadata_analysis = analyze_page_metadata(pages_df)
    print(json.dumps(metadata_analysis, indent=2, default=str))
else:
    print("No page data available. Run the page extraction cell first.")

{
  "total_pages": 8,
  "page_versions": {
    "min": "0",
    "max": "0",
    "avg": 0.0
  }
}


## Visualize Page Activity

Let's create some visualizations of page activity over time.

In [13]:
# Visualization of page updates over time
if 'pages_df' in locals() and not pages_df.empty and 'updated' in pages_df.columns:
    # Make sure the updated column is datetime type
    if pd.api.types.is_datetime64_any_dtype(pages_df['updated']):
        # Group by month and count updates
        pages_df['update_month'] = pages_df['updated'].dt.to_period('M')
        monthly_counts = pages_df.groupby('update_month').size().reset_index(name='count')
        monthly_counts['update_month_str'] = monthly_counts['update_month'].astype(str)
        
        # Plot using pandas
        ax = monthly_counts.plot(x='update_month_str', y='count', kind='bar', figsize=(12, 6))
        ax.set_title(f'Page Updates by Month in Space "{SPACE_KEY}"')
        ax.set_xlabel('Month')
        ax.set_ylabel('Number of Page Updates')
        
        # Rotate x-axis labels for better readability
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("The 'updated' column is not in datetime format.")
else:
    print("No page data available or 'updated' column missing. Run the page extraction cell first.")

The 'updated' column is not in datetime format.


## Content Analysis

Perform text analysis on the extracted page content.

## Export Data

Export the extracted data for further processing or backup.

In [13]:
# Export data to different formats
def export_to_csv(df, filename):
    """Export DataFrame to CSV file"""
    try:
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"Data successfully exported to {filename}")
        return True
    except Exception as e:
        print(f"Error exporting to CSV: {e}")
        return False

def export_to_json(data, filename):
    """Export data to JSON file"""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
        print(f"Data successfully exported to {filename}")
        return True
    except Exception as e:
        print(f"Error exporting to JSON: {e}")
        return False

# Export page metadata
if 'pages_df' in locals() and not pages_df.empty:
    # Create an export directory if it doesn't exist
    os.makedirs('exports', exist_ok=True)
    
    # Timestamp for unique filenames
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Export page metadata to CSV
    csv_filename = f"exports/confluence_pages_{SPACE_KEY}_{timestamp}.csv"
    export_to_csv(pages_df, csv_filename)
    
    # Export full content data if available
    if 'content_df' in locals() and not content_df.empty:
        # For CSV, only export metadata columns (no HTML content)
        metadata_cols = [col for col in content_df.columns if col not in ['html_content', 'plain_text']]
        content_csv_filename = f"exports/confluence_content_metadata_{SPACE_KEY}_{timestamp}.csv"
        export_to_csv(content_df[metadata_cols], content_csv_filename)
        
        # For JSON, export everything including content
        content_json_filename = f"exports/confluence_full_content_{SPACE_KEY}_{timestamp}.json"
        export_to_json(content_df.to_dict(orient='records'), content_json_filename)
else:
    print("No data available to export. Run the extraction first.")

Data successfully exported to exports/confluence_pages_Founders_20250503_140439.csv
Data successfully exported to exports/confluence_content_metadata_Founders_20250503_140439.csv
Data successfully exported to exports/confluence_full_content_Founders_20250503_140439.json


## Next Steps

Here are some suggestions for further analysis or data extraction:

1. Extract and analyze attachments from pages
2. Build a network graph of page relationships (parent-child)
3. Perform more advanced text analysis (NLP) on page content
4. Extract tables from pages and analyze structured data
5. Implement scheduled extractions for tracking changes over time

This notebook provides a foundation that you can extend for your specific use cases.

## Direct Data Extraction to Dataset

This section simplifies the process of extracting Confluence data directly into a pandas DataFrame.

In [ ]:
def extract_confluence_data_to_dataset(space_key, include_content=False, max_pages=None):
    """Extract data from Confluence directly to a pandas DataFrame
    
    Parameters:
    -----------
    space_key : str
        The Confluence space key to extract data from
    include_content : bool, default=False
        Whether to include page content in the dataset (significantly increases size)
    max_pages : int, optional
        Maximum number of pages to extract (None for all pages)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame containing the extracted Confluence data
    """
    print(f"Extracting data from Confluence space: {space_key}")
    
    # Create session
    try:
        session = create_confluence_session()
        print("✓ Connection established")
    except Exception as e:
        print(f"Error creating session: {e}")
        return pd.DataFrame()
    
    # Get pages
    print("Retrieving pages...")
    pages = get_pages_in_space(session, space_key)
    if not pages:
        print(f"No pages found in space '{space_key}'")
        return pd.DataFrame()
        
    total_pages = len(pages)
    print(f"Found {total_pages} pages in space '{space_key}")
    
    # Limit pages if specified
    if max_pages and max_pages < total_pages:
        pages = pages[:max_pages]
        print(f"Limiting to {max_pages} pages")
    
    # Prepare data collection
    all_data = []
    
    # Extract page data
    for i, page in enumerate(pages, 1):
        page_id = page['id']
        title = page['title']
        print(f"Processing page {i}/{len(pages)}: {title}")
        
        page_data = {
            'id': page_id,
            'title': title,
            'type': page.get('type', 'unknown'),
            'status': page.get('status', 'unknown'),
            'created': page.get('history', {}).get('createdDate', 'unknown'),
            'updated': page.get('history', {}).get('lastUpdated', {}).get('when', 'unknown'),
            'version': page.get('version', {}).get('number', 0),
            'url': f"{CONFLUENCE_URL}/wiki/spaces/{space_key}/pages/{page_id}"
        }
        
        # Get content if requested
        if include_content:
            page_detail = get_page_content(session, page_id, expand='body.storage,version,ancestors,metadata')
            
            if page_detail and 'body' in page_detail and 'storage' in page_detail['body']:
                html_content = page_detail['body']['storage']['value']
                page_data['html_content'] = html_content
                page_data['plain_text'] = extract_text_from_html(html_content)
                
                # Get parent page hierarchy
                ancestors = [a.get('title', 'Unknown') for a in page_detail.get('ancestors', [])]
                page_data['ancestors'] = ' > '.join(ancestors) if ancestors else 'Root'
                
                # Get labels/tags if available
                if 'metadata' in page_detail and 'labels' in page_detail['metadata']:
                    labels = [label.get('name', '') for label in page_detail['metadata']['labels'].get('results', [])]
                    page_data['labels'] = ', '.join(labels) if labels else ''
            else:
                print(f"  ⚠️ Could not retrieve content for {title}")
        
        all_data.append(page_data)
        time.sleep(0.5)  # Be nice to the API
    
    # Convert to DataFrame
    df = pd.DataFrame(all_data)
    
    # Convert date columns to datetime
    for col in ['created', 'updated']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='ignore')
    
    print(f"✓ Successfully extracted data for {len(df)} pages")
    return df

## Extract Data and Export

Run this cell to extract Confluence data and save it to CSV/Excel.

In [15]:
# Set your space key here
SPACE_KEY = "Founders"  # Replace with the actual space key

# Extract data into a dataset
confluence_dataset = extract_confluence_data_to_dataset(
    space_key=SPACE_KEY,
    include_content=True,  # Set to False if you only need metadata
    max_pages=None  # Set a number to limit pages, or None for all pages
)

# Display the first few rows of the dataset
if not confluence_dataset.empty:
    # Display basic info about the dataset
    print(f"\nDataset shape: {confluence_dataset.shape} (rows × columns)")
    print(f"Columns: {list(confluence_dataset.columns)}")
    
    # Show the dataframe
    display(confluence_dataset.head())
    
    # Create exports directory
    os.makedirs('exports', exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Define export paths
    csv_path = f"exports/confluence_data_{SPACE_KEY}_{timestamp}.csv"
    excel_path = f"exports/confluence_data_{SPACE_KEY}_{timestamp}.xlsx"
    
    # Export to CSV
    try:
        # For CSV exclude the HTML content column to avoid issues
        export_cols = [col for col in confluence_dataset.columns if col != 'html_content']
        confluence_dataset[export_cols].to_csv(csv_path, index=False, encoding='utf-8')
        print(f"✓ Data exported to CSV: {csv_path}")
    except Exception as e:
        print(f"Error exporting to CSV: {e}")
    
    # Export to Excel
    try:
        writer = pd.ExcelWriter(excel_path, engine='openpyxl')
        confluence_dataset.to_excel(writer, index=False, sheet_name='Confluence Data')
        writer.close()
        print(f"✓ Data exported to Excel: {excel_path}")
    except Exception as e:
        print(f"Error exporting to Excel: {e}")
else:
    print("No data was extracted. Please check your space key and connection settings.")

Extracting data from Confluence space: Founders
✓ Connection established
Retrieving pages...
Found 8 pages in space 'Founders'
Processing page 1/8: Founders
Found 8 pages in space 'Founders'
Processing page 1/8: Founders
Processing page 2/8: Go To Market Sales Playbook
Processing page 2/8: Go To Market Sales Playbook
Processing page 3/8: Branding, Marketing, & Investors
Processing page 3/8: Branding, Marketing, & Investors
Processing page 4/8: Future Ideas of OhmGym
Processing page 4/8: Future Ideas of OhmGym
Processing page 5/8: Angel Investor Pitch Notes
Processing page 5/8: Angel Investor Pitch Notes
Processing page 6/8: Business Plan
Processing page 6/8: Business Plan
Processing page 7/8: Competitors Systems and Services
Processing page 7/8: Competitors Systems and Services
Processing page 8/8: Google Ads Metrics
Processing page 8/8: Google Ads Metrics
✓ Successfully extracted data for 8 pages

Dataset shape: (8, 11) (rows × columns)
Columns: ['id', 'title', 'type', 'status', 'crea

/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1348114151.py:92: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_datetime(df[col], errors='ignore')
/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1348114151.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='ignore')
/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1348114151.py:92: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_datetime(df[col], errors='ignore')
/var/folders/49/l0jy6fgd39s5v_9246d2k6440000gn/T/ipykernel_45288/1348114151.py:92: U

,id,title,type,status,created,updated,version,url,html_content,plain_text,ancestors
0,4980952,Founders,page,current,unknown,unknown,0,https://ohmgym.atlassian.net/wiki/spaces/Found...,"<ac:layout><ac:layout-section ac:type=""fixed-w...",:check_mark:atlassian-check_mark#FFF0B3\n\n## ...,Root
1,5242881,Go To Market Sales Playbook,page,current,unknown,unknown,0,https://ohmgym.atlassian.net/wiki/spaces/Found...,<p>Step 1: Identify Ideal Customer Profile - I...,Step 1: Identify Ideal Customer Profile - ICP\...,Founders
2,5308428,"Branding, Marketing, & Investors",page,current,unknown,unknown,0,https://ohmgym.atlassian.net/wiki/spaces/Found...,<p><strong>Branding</strong></p><ul><li><p>We ...,**Branding**\n\n * We are looking to build a ...,Founders
3,5341200,Future Ideas of OhmGym,page,current,unknown,unknown,0,https://ohmgym.atlassian.net/wiki/spaces/Found...,<ul><li><p>Filter features for price limits or...,* Filter features for price limits or ceilin...,Founders
4,5341213,Angel Investor Pitch Notes,page,current,unknown,unknown,0,https://ohmgym.atlassian.net/wiki/spaces/Found...,<p>We believe we can deliver value by unlockin...,We believe we can deliver value by unlocking u...,Founders


✓ Data exported to CSV: exports/confluence_data_Founders_20250503_140454.csv
Error exporting to Excel: No module named 'openpyxl'
